# Apples

### Import

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from arch import arch_model 

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

### Data 

In [ ]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

### SPREAD

In [ ]:
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

### ENGINE

In [ ]:
df['Date'] = df.index.date
unique_days = df['Date'].unique()

train_days = 30 
out_of_sample_results = []
parameter_tracker = [] # NEW: We will store the daily parameters here

# The Walk-Forward Loop
for i in range(train_days, len(unique_days)):
    train_dates = unique_days[i - train_days : i]
    test_date = unique_days[i]
    
    train_df = df[df['Date'].isin(train_dates)].copy()
    test_df = df[df['Date'] == test_date].copy()
    
    # Fit Models
    engine = ENGINE(train_df)
    train_fitted = engine.fit_cointegration(z_window=1000)
    engine.fit_ar_reversion(lags=1)
    engine.fit_garch_vol(scaling=10000)
    engine.fit_markov_regimes(k_regimes=2)
    
    # Predict Out-of-Sample
    oos_predictions = engine.predict_oos(test_df, train_fitted, z_window=1000)
    out_of_sample_results.append(oos_predictions)
    
    # --- NEW: TRACK THE DAILY PARAMETERS ---
    parameter_tracker.append({
        'Date': test_date,
        'Beta': engine.beta,
        'Alpha': engine.alpha,
        'Safe_Variance': engine.safe_variance,
        'Danger_Variance': engine.danger_variance,
        'GARCH_Vol': engine.forecasted_vol,
        'AR_Phi': engine.ar_phi
    })

# Stitch data together
live_trading_data = pd.concat(out_of_sample_results)
df_params = pd.DataFrame(parameter_tracker).set_index('Date') # Convert tracker to DataFrame

print(f"\n OOS Dataset Built: {len(live_trading_data)} rows.")
print(" Parameter tracking complete.")

### BACKTEST

In [ ]:
bt = BACKTESTER(live_trading_data)

# Try a slightly higher Danger Threshold (0.6) and the new AR limit
results_df = bt.run(
    base_z=2.0, 
    danger_threshold=0.6, 
    ar_limit=0.9, 
    fee_bps=0.5
)

### TEARSHEEET

In [ ]:
ts = TEARSHEET(results_df)
ts.calculate_financials()
ts.plot_performance()